# `run.ipynb` — universal stage runner

**What this does**
1. Mounts Google Drive (Colab only) so the cache + outputs persist across sessions. Locally, everything just lives on disk in this repo checkout.
2. Sources every code cell from `lib.ipynb` (no copy-paste — single source of truth).
3. Reads one config YAML and calls `run_stage(...)`.
4. Saves outputs under `<output_dir>/<stage_id>/` (config snapshot, predictions, metrics, plots, summary).

**To run on Colab**
- Upload this repo's `forecasting/` contents + `data_pipeline/data/processed/...` to `MyDrive/moex-hack/`, mirroring this repo's local directory layout.
- Edit the `CONFIG_PATH` cell below.
- Runtime → Run all.

**To run locally (no Colab, no Drive)**
- Open this notebook from within a Jupyter/VS Code kernel — §0 auto-detects it's not Colab and resolves `PROJECT_DIR` to wherever `lib.ipynb` actually is (works whether the kernel's cwd is `forecasting/` or the repo root).
- Needs `chronos-forecasting`, `pandas[pyarrow]`, `requests`, `matplotlib`, `numpy`, `tqdm`, `pyyaml`, `scipy`, `nbformat` installed in the local Python environment (same list §1's Colab `!pip install` cell uses — install them yourself first, that cell is Colab-only).
- No GPU needed for this to *work* — `lib.ipynb` §1 already falls back to `DEVICE="cpu"`/`DTYPE=torch.float32` automatically — but every `predict_df` call is a real forward pass through the model, and the basket-gate configs run hundreds of them (`max_windows: 400` × 2 arms), so expect this to take meaningfully longer than on a GPU. Time a handful of windows first if you want a realistic ETA before committing to a full run.
- `algopack_processed_path` in the basket-gate configs resolves automatically for both layouts (flat Colab-Drive layout, or this repo's `forecasting/` + `../data_pipeline/` local layout) — see `load_config` in `lib.ipynb` §2.

**To run in parallel**
- Open a second Colab session, same notebook file, point `CONFIG_PATH` at a different config. Cache is shared (Drive); outputs are namespaced by `stage_id`.

**First time only**
- Run the optional "bulk prefetch" cell once to fill the cache — softer on ISS than letting every run prefetch its own slice on first run.

**Note**: an earlier numbered-stage config sequence (stage_0 through stage_7) preceded the current gated-experiment configs (`configs/`) and is not included in this repo.

## 0. Locate the project

On Colab: set `PROJECT_DIR` to the folder containing this notebook + `lib.ipynb` + `configs/` (default assumes `MyDrive/moex-hack/`). Locally: auto-detected below — resolves to wherever `lib.ipynb` actually sits, so it works whether you launched the kernel from `forecasting/` or the repo root. If it still gets it wrong (unusual folder layout), set `PROJECT_DIR` by hand before running the next cell.

In [ ]:
import os
from pathlib import Path

IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in str(type(globals().get("get_ipython", lambda: None)()))
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive", force_remount=False)
    PROJECT_DIR = "/content/drive/MyDrive/moex-hack"
else:
    # Local run: PROJECT_DIR must be the folder containing lib.ipynb + configs/
    # (i.e. forecasting/ in this repo). Path.cwd() only gives the right answer if the
    # kernel was launched with forecasting/ as the working directory -- if this notebook
    # was opened from elsewhere (repo root, a different folder), fall back to
    # resolving relative to this .ipynb file's own location instead of guessing.
    candidates = [Path.cwd(), Path.cwd() / "forecasting"]
    PROJECT_DIR = next((str(c) for c in candidates if (c / "lib.ipynb").exists()), str(Path.cwd()))

os.chdir(PROJECT_DIR)
print(f"PROJECT_DIR = {PROJECT_DIR}")
print("contents:", sorted(os.listdir(PROJECT_DIR)))
if not IN_COLAB and not (Path(PROJECT_DIR) / "lib.ipynb").exists():
    print("WARNING: lib.ipynb not found here -- set PROJECT_DIR manually to "
          "the folder that contains it (locally: the repo's forecasting/ folder) before "
          "continuing to \xa71.")


## 1. Source `lib.ipynb`

Executes every code cell from the cell library inside this kernel — no imports, no module packaging, no version drift.

In [ ]:
import json, nbformat
from IPython import get_ipython

BASIC = Path(PROJECT_DIR) / "lib.ipynb"
assert BASIC.exists(), f"missing {BASIC} — copy it next to this notebook"
nb = nbformat.read(str(BASIC), as_version=4)
ip = get_ipython()
n_code = 0
for cell in nb.cells:
    if cell.cell_type == "code":
        ip.run_cell(cell.source)
        n_code += 1
print(f"sourced {n_code} code cells from lib.ipynb")


## 2. Data prefetch and discovery/confirmation screens

Run once on a fresh Drive cache. Idempotent — safe to re-run.

In [ ]:
# Uncomment to bulk-prefetch every configured run's data into the cache.
all_cfgs = [load_config(str(p)) for p in sorted(Path("configs").glob("*.yaml"))]
cache_dir = resolve_cache_dir(all_cfgs[0])
manifest = build_prefetch_manifest(all_cfgs)
print(f"manifest size: {len(manifest)} files")
prefetch_all(manifest, cache_dir)

### 2.1 Lead-lag discovery screen (no Chronos, pandas only)

Runs `pairwise_lagged_xcorr`/`select_pair_shortlist` (§14 of `lib.ipynb`)
against the discovery-window slice (2020-01-03..2023-06-30) of the 80-ticker
AlgoPack pull. Independent of §3/§4 below — does not call `run_stage`, no
model load, seconds not minutes. Run this before touching `leadlag_confirm_adj.yaml`.

In [ ]:
DISCOVERY_INTERVAL = 24   # 24 = daily, 60 = 1h (edit me for the 1h follow-on)
DISCOVERY_FROM, DISCOVERY_TILL = "2020-01-03", "2023-06-30"
DISCOVERY_MIN_COVERAGE = 0.9
DISCOVERY_PRICE_COL = "close_adj"   # the original daily/1h discovery ran on raw
# "close"; set to "close" to reproduce that. For the 1h follow-on, set
#   DISCOVERY_INTERVAL, DISCOVERY_FROM, DISCOVERY_TILL = 60, "2023-01-02", "2024-05-24"
# (24mo window 2023-01-02..2024-12-30, ~70/30 split: 365 discovery trading days / 156
# confirmation trading days, verified zero overlap. Confirmation config
# leadlag_confirm_1h_adj.yaml uses date_from="2024-05-27" -- keep both in sync.)

_INTERVAL_PATHS = {24: "data_pipeline/data/processed/candles_1d/shares.parquet",
                    60: "data_pipeline/data/processed/candles_1h/shares.parquet"}

import yaml as _yaml
_algo_cfg_text = (Path("..") / "data_pipeline" / "config.md").read_text()
_algo_cfg = _yaml.safe_load(_algo_cfg_text.split("```yaml")[1].split("```")[0])
DISCOVERY_TICKERS = sorted(_algo_cfg["tickers"]["shares"])
print(f"{len(DISCOVERY_TICKERS)} tickers configured in data_pipeline/config.md")

# load_config resolves the flat-vs-nested algopack_processed_path fallback for us --
# reuse it with a minimal stand-in cfg rather than hand-rolling path logic. indexes/
# futures_proxies are explicitly empty: this step only needs `prices` (from the
# algopack parquet), and a non-empty indexes list would otherwise make
# load_stage_inputs try ISS cache lookups we don't want here.
_stub_cfg_path = Path("configs/_discovery_stub.yaml")
_stub_cfg_path.write_text(_yaml.safe_dump(dict(
    stage_id="leadlag_discovery_stub", interval=DISCOVERY_INTERVAL, tickers=DISCOVERY_TICKERS,
    date_from=DISCOVERY_FROM, date_till=DISCOVERY_TILL,
    context_len=250, horizon=5, walk_forward=dict(shift=1, max_windows=400),
    covariates="none", output_dir="runs",
    data_source="algopack",
    algopack_processed_path=_INTERVAL_PATHS[DISCOVERY_INTERVAL],
    min_ticker_coverage=DISCOVERY_MIN_COVERAGE, indexes=[], futures_proxies=[],
    price_col=DISCOVERY_PRICE_COL,
)))
discovery_cfg = load_config(str(_stub_cfg_path))
_stub_cfg_path.unlink()  # scratch file, not a real stage config -- delete right after loading

prices, _, _ = load_stage_inputs(discovery_cfg, cache_dir=None)
price_panel = build_price_panel(prices, interval=DISCOVERY_INTERVAL, value_col=DISCOVERY_PRICE_COL, min_ticker_coverage=DISCOVERY_MIN_COVERAGE)
# load_from_algopack/load_stage_inputs don't filter by date on their own (that normally
# happens later inside assemble_panels/run_stage) -- since we're bypassing run_stage for
# this pandas-only step, slice to the discovery window explicitly here.
price_panel = price_panel.loc[DISCOVERY_FROM:DISCOVERY_TILL]
ret_panel = log_returns(price_panel)
print(f"discovery panel: {ret_panel.shape[0]} bars x {ret_panel.shape[1]} tickers "
      f"({ret_panel.index.min()} .. {ret_panel.index.max()})")

xcorr, shortlist = select_pair_shortlist(pairwise_lagged_xcorr(ret_panel, max_lag=5), alpha=0.05, top_n=20)
print(f"{len(xcorr)} (pair, lag, direction) tests run, {len(shortlist)} BH-significant (q<0.05, top-20 cap)")
shortlist

### 2.2 Event-conditioned burst discovery (no Chronos, pandas only)

Runs `run_e2_discovery` (§16 of `lib.ipynb`) against the SAME 1h discovery
window already used for the lead-lag confirmation screen's 1h follow-on (2023-01-02..2024-05-24) — reuses the
date split, not that screen's shortlist (a different mechanism, see §16's header
comment for why). Requires §1 to have run first (sources `lib.ipynb`,
including E1's synthetic validation — **if E1's assertions fail, this cell's
functions won't exist correctly and you should stop and investigate rather than
run E2 on unverified code**). Independent of §2.1/§3/§4 below.


In [ ]:
E2_INTERVAL = 60
E2_DISCOVERY_FROM, E2_DISCOVERY_TILL = "2023-01-02", "2024-05-24"
E2_CONFIRM_FROM, E2_CONFIRM_TILL = "2024-05-27", "2024-12-30"
E2_MIN_COVERAGE = 0.9
E2_LAGS = (1, 2, 3, 4)
E2_THRESHOLD_STD = 2.0
E2_MIN_EVENTS = 85  # derived floor -- see lib.ipynb §16 header for the power
                    # calculation this comes from; do not lower without recomputing it

import yaml as _yaml
_algo_cfg_text = (Path("..") / "data_pipeline" / "config.md").read_text()
_algo_cfg = _yaml.safe_load(_algo_cfg_text.split("```yaml")[1].split("```")[0])
E2_TICKERS = sorted(_algo_cfg["tickers"]["shares"])

_stub_cfg_path = Path("configs/_e2_discovery_stub.yaml")
_stub_cfg_path.write_text(_yaml.safe_dump(dict(
    stage_id="phase_e2_discovery_stub", interval=E2_INTERVAL, tickers=E2_TICKERS,
    date_from=E2_DISCOVERY_FROM, date_till=E2_DISCOVERY_TILL,
    context_len=250, horizon=5, walk_forward=dict(shift=1, max_windows=400),
    covariates="none", output_dir="runs",
    data_source="algopack",
    algopack_processed_path="data_pipeline/data/processed/candles_1h/shares.parquet",
    min_ticker_coverage=E2_MIN_COVERAGE, indexes=[], futures_proxies=[],
)))
e2_discovery_cfg = load_config(str(_stub_cfg_path))
_stub_cfg_path.unlink()

e2_prices, _, _ = load_stage_inputs(e2_discovery_cfg, cache_dir=None)
e2_price_panel_full = build_price_panel(e2_prices, interval=E2_INTERVAL, min_ticker_coverage=E2_MIN_COVERAGE)

e2_disc_price_panel = e2_price_panel_full.loc[E2_DISCOVERY_FROM:E2_DISCOVERY_TILL]
e2_disc_ret_panel = log_returns(e2_disc_price_panel)
print(f"E2 discovery panel: {e2_disc_ret_panel.shape[0]} bars x {e2_disc_ret_panel.shape[1]} tickers "
      f"({e2_disc_ret_panel.index.min()} .. {e2_disc_ret_panel.index.max()})")

e2_scan, e2_shortlist, e2_coverage = run_e2_discovery(
    e2_disc_ret_panel, lags=E2_LAGS, threshold_std=E2_THRESHOLD_STD, min_events=E2_MIN_EVENTS,
)
print(f"\nE2 discovery coverage: {e2_coverage}")
print(f"{len(e2_shortlist)} BH-significant (leader, follower, lag) hypotheses (q<0.05)")
e2_shortlist


### 2.3 Event-conditioned burst confirmation (no Chronos, pandas only)

Requires §2.2 to have run first (`e2_shortlist` must exist). If `e2_shortlist` is
empty, this is already a complete, valid result — per the same pre-registered rule
used throughout this project (0 survivors is not a trigger to loosen the threshold).
Re-tests each discovery hypothesis on the INDEPENDENT confirmation window
(2024-05-27..2024-12-30, same split as the lead-lag confirmation screen's 1h follow-on), own BH correction
within this smaller family, never reusing discovery p-values.


In [ ]:
assert "e2_shortlist" in dir(), "run §2.2 first -- e2_shortlist is not defined"

if e2_shortlist.empty:
    print("E2 discovery shortlist is EMPTY -- 0 candidate hypotheses to confirm.")
    print("This is a complete, valid result per the pre-registered rule (do not loosen "
          "the discovery threshold and re-run). Write up and stop.")
    e2_confirm = e2_shortlist.assign(p_value_bh_confirm=[], confirmed=[])
else:
    e2_confirm_price_panel = e2_price_panel_full.loc[E2_CONFIRM_FROM:E2_CONFIRM_TILL]
    e2_confirm_ret_panel = log_returns(e2_confirm_price_panel)
    print(f"E2 confirmation panel: {e2_confirm_ret_panel.shape[0]} bars x "
          f"{e2_confirm_ret_panel.shape[1]} tickers "
          f"({e2_confirm_ret_panel.index.min()} .. {e2_confirm_ret_panel.index.max()})")

    e2_confirm = run_e2_confirmation(
        e2_confirm_ret_panel, e2_shortlist, lags=E2_LAGS, threshold_std=E2_THRESHOLD_STD,
        min_events=E2_MIN_EVENTS,
    )
    n_confirmed = int(e2_confirm["confirmed"].sum())
    print(f"\n{len(e2_confirm)} discovery hypotheses re-tested, {n_confirmed} confirmed "
          f"(BH q<0.05 within this confirmation family)")

e2_confirm


### 2.4 Event-conditioned burst discovery — daily follow-on

Same mechanism as §2.2/§2.3 (`run_e2_discovery`/`run_e2_confirmation`, §16 of
`lib.ipynb`) but at DAILY resolution with parameters re-derived for daily's
much lower bar count — **not** a copy of the 1h parameters. Daily has only 1249 bars
total (2020-2024) vs. 1h's ~17600; a 2σ leader-event threshold gives too few events
per ticker to reach any reasonable floor (~9-16 events in a 1h-sized window, ~56 over
the full history — well under even 1h's n≥85 floor).

**Re-derived design (locked in before running)**:
- `threshold_std=1.25` (vs. 1h's 2.0) — a real "notable move" threshold, looser than
  a strict 2σ shock but still meaningfully above ordinary daily noise (a 1σ move
  happens ~32% of days, which would blur into "any day"; 1.25σ happens ~21% of days).
- 70/30 chronological split of the full 2020-2024 history (no shorter sub-window —
  daily doesn't have enough bars to spare): discovery 2020-01-03..2023-07-17 (874
  bars), confirmation 2023-07-18..2024-12-30 (375 bars). Verified zero overlap.
- Floor **n≥79**, set by the CONFIRMATION window's achievable event count (~79
  expected events at threshold_std=1.25 over 375 bars), not discovery's — the
  confirmation window is the binding constraint here (it's the smaller of the two,
  unlike 1h where both sides had comfortable headroom). Derived via the same
  binomial two-proportion power calculation as 1h's floor, targeting an ~66%
  same-direction effect at 80% power/α=0.05 (close to 1h's 65% target, not
  identical, since daily's smaller confirmation budget sets a slightly higher
  detectable-effect floor).
- Lags 1-4 **trading days** (not bars/hours) — the same economic story as the daily
  lead-lag confirmation screen's lag range, not 1h's intraday-propagation story.

This is a genuinely different, freshly-derived test, not a resolution swap on the
same numbers — the confirmation window's achievable event count was the binding
constraint on the floor, unlike 1h where both sides had comfortable headroom.


In [ ]:
E2D_INTERVAL = 24
E2D_DISCOVERY_FROM, E2D_DISCOVERY_TILL = "2020-01-03", "2023-07-17"
E2D_CONFIRM_FROM, E2D_CONFIRM_TILL = "2023-07-18", "2024-12-30"
E2D_MIN_COVERAGE = 0.9
E2D_LAGS = (1, 2, 3, 4)
E2D_THRESHOLD_STD = 1.25  # re-derived for daily -- see §2.4 markdown, NOT the 1h value
E2D_MIN_EVENTS = 79       # re-derived for daily, set by confirmation's smaller budget

import yaml as _yaml
_algo_cfg_text = (Path("..") / "data_pipeline" / "config.md").read_text()
_algo_cfg = _yaml.safe_load(_algo_cfg_text.split("```yaml")[1].split("```")[0])
E2D_TICKERS = sorted(_algo_cfg["tickers"]["shares"])

_stub_cfg_path = Path("configs/_e2d_discovery_stub.yaml")
_stub_cfg_path.write_text(_yaml.safe_dump(dict(
    stage_id="phase_e2_daily_discovery_stub", interval=E2D_INTERVAL, tickers=E2D_TICKERS,
    date_from=E2D_DISCOVERY_FROM, date_till=E2D_DISCOVERY_TILL,
    context_len=250, horizon=5, walk_forward=dict(shift=1, max_windows=400),
    covariates="none", output_dir="runs",
    data_source="algopack",
    algopack_processed_path="data_pipeline/data/processed/candles_1d/shares.parquet",
    min_ticker_coverage=E2D_MIN_COVERAGE, indexes=[], futures_proxies=[],
)))
e2d_discovery_cfg = load_config(str(_stub_cfg_path))
_stub_cfg_path.unlink()

e2d_prices, _, _ = load_stage_inputs(e2d_discovery_cfg, cache_dir=None)
e2d_price_panel_full = build_price_panel(e2d_prices, interval=E2D_INTERVAL, min_ticker_coverage=E2D_MIN_COVERAGE)

e2d_disc_price_panel = e2d_price_panel_full.loc[E2D_DISCOVERY_FROM:E2D_DISCOVERY_TILL]
e2d_disc_ret_panel = log_returns(e2d_disc_price_panel)
print(f"E2 (daily) discovery panel: {e2d_disc_ret_panel.shape[0]} bars x "
      f"{e2d_disc_ret_panel.shape[1]} tickers "
      f"({e2d_disc_ret_panel.index.min()} .. {e2d_disc_ret_panel.index.max()})")

e2d_scan, e2d_shortlist, e2d_coverage = run_e2_discovery(
    e2d_disc_ret_panel, lags=E2D_LAGS, threshold_std=E2D_THRESHOLD_STD, min_events=E2D_MIN_EVENTS,
)
print(f"\nE2 (daily) discovery coverage: {e2d_coverage}")
print(f"{len(e2d_shortlist)} BH-significant (leader, follower, lag) hypotheses (q<0.05)")
e2d_shortlist


### 2.5 Event-conditioned burst confirmation — daily follow-on

Requires §2.4 to have run first (`e2d_shortlist` must exist). If empty, this is
already a complete, valid result per the same pre-registered rule used throughout
this project.


In [ ]:
assert "e2d_shortlist" in dir(), "run §2.4 first -- e2d_shortlist is not defined"

if e2d_shortlist.empty:
    print("E2 (daily) discovery shortlist is EMPTY -- 0 candidate hypotheses to confirm.")
    print("This is a complete, valid result per the pre-registered rule (do not loosen "
          "the discovery threshold and re-run). Write up and stop.")
    e2d_confirm = e2d_shortlist.assign(p_value_bh_confirm=[], confirmed=[])
else:
    e2d_confirm_price_panel = e2d_price_panel_full.loc[E2D_CONFIRM_FROM:E2D_CONFIRM_TILL]
    e2d_confirm_ret_panel = log_returns(e2d_confirm_price_panel)
    print(f"E2 (daily) confirmation panel: {e2d_confirm_ret_panel.shape[0]} bars x "
          f"{e2d_confirm_ret_panel.shape[1]} tickers "
          f"({e2d_confirm_ret_panel.index.min()} .. {e2d_confirm_ret_panel.index.max()})")

    e2d_confirm = run_e2_confirmation(
        e2d_confirm_ret_panel, e2d_shortlist, lags=E2D_LAGS, threshold_std=E2D_THRESHOLD_STD,
        min_events=E2D_MIN_EVENTS,
    )
    n_confirmed = int(e2d_confirm["confirmed"].sum())
    print(f"\n{len(e2d_confirm)} discovery hypotheses re-tested, {n_confirmed} confirmed "
          f"(BH q<0.05 within this confirmation family)")

e2d_confirm


### 2.6 Event-conditioned burst discovery — daily, loosened threshold

Same daily split/panel as §2.4, but `threshold_std=1.0` (down from §2.4's 1.25) and
floor n≥90 (up from 79, since the lower threshold generates more events) — see
this section's own scope: §2.4/§2.5's 3
discovery candidates could not be adequately re-tested in confirmation (all 3 fell
below the n≥79 floor there), so this is a genuinely fresh, more powerful re-run,
not a re-check of the same 3 candidates under a looser bar (that would not be a
valid test — we'd already seen their confirmation-window numbers).


In [ ]:
E2D2_INTERVAL = 24
E2D2_DISCOVERY_FROM, E2D2_DISCOVERY_TILL = "2020-01-03", "2023-07-17"
E2D2_CONFIRM_FROM, E2D2_CONFIRM_TILL = "2023-07-18", "2024-12-30"
E2D2_MIN_COVERAGE = 0.9
E2D2_LAGS = (1, 2, 3, 4)
E2D2_THRESHOLD_STD = 1.0  # loosened from §2.4's 1.25 -- see §2.6 markdown
E2D2_MIN_EVENTS = 90      # re-derived for the loosened threshold's higher event rate

import yaml as _yaml
_algo_cfg_text = (Path("..") / "data_pipeline" / "config.md").read_text()
_algo_cfg = _yaml.safe_load(_algo_cfg_text.split("```yaml")[1].split("```")[0])
E2D2_TICKERS = sorted(_algo_cfg["tickers"]["shares"])

_stub_cfg_path = Path("configs/_e2d2_discovery_stub.yaml")
_stub_cfg_path.write_text(_yaml.safe_dump(dict(
    stage_id="phase_e2_daily_v2_discovery_stub", interval=E2D2_INTERVAL, tickers=E2D2_TICKERS,
    date_from=E2D2_DISCOVERY_FROM, date_till=E2D2_DISCOVERY_TILL,
    context_len=250, horizon=5, walk_forward=dict(shift=1, max_windows=400),
    covariates="none", output_dir="runs",
    data_source="algopack",
    algopack_processed_path="data_pipeline/data/processed/candles_1d/shares.parquet",
    min_ticker_coverage=E2D2_MIN_COVERAGE, indexes=[], futures_proxies=[],
)))
e2d2_discovery_cfg = load_config(str(_stub_cfg_path))
_stub_cfg_path.unlink()

e2d2_prices, _, _ = load_stage_inputs(e2d2_discovery_cfg, cache_dir=None)
e2d2_price_panel_full = build_price_panel(e2d2_prices, interval=E2D2_INTERVAL, min_ticker_coverage=E2D2_MIN_COVERAGE)

e2d2_disc_price_panel = e2d2_price_panel_full.loc[E2D2_DISCOVERY_FROM:E2D2_DISCOVERY_TILL]
e2d2_disc_ret_panel = log_returns(e2d2_disc_price_panel)
print(f"E2 (daily v2) discovery panel: {e2d2_disc_ret_panel.shape[0]} bars x "
      f"{e2d2_disc_ret_panel.shape[1]} tickers "
      f"({e2d2_disc_ret_panel.index.min()} .. {e2d2_disc_ret_panel.index.max()})")

e2d2_scan, e2d2_shortlist, e2d2_coverage = run_e2_discovery(
    e2d2_disc_ret_panel, lags=E2D2_LAGS, threshold_std=E2D2_THRESHOLD_STD, min_events=E2D2_MIN_EVENTS,
)
print(f"\nE2 (daily v2) discovery coverage: {e2d2_coverage}")
print(f"{len(e2d2_shortlist)} BH-significant (leader, follower, lag) hypotheses (q<0.05)")
e2d2_shortlist


### 2.7 Event-conditioned burst confirmation — daily, loosened threshold

Requires §2.6 to have run first (`e2d2_shortlist` must exist). If empty, this is
already a complete, valid result — confirmation not needed.


In [ ]:
assert "e2d2_shortlist" in dir(), "run §2.6 first -- e2d2_shortlist is not defined"

if e2d2_shortlist.empty:
    print("E2 (daily v2) discovery shortlist is EMPTY -- 0 candidate hypotheses to confirm.")
    print("This is a complete, valid result per the pre-registered rule (do not loosen "
          "the discovery threshold and re-run again without a fresh justification).")
    e2d2_confirm = e2d2_shortlist.assign(p_value_bh_confirm=[], confirmed=[])
else:
    e2d2_confirm_price_panel = e2d2_price_panel_full.loc[E2D2_CONFIRM_FROM:E2D2_CONFIRM_TILL]
    e2d2_confirm_ret_panel = log_returns(e2d2_confirm_price_panel)
    print(f"E2 (daily v2) confirmation panel: {e2d2_confirm_ret_panel.shape[0]} bars x "
          f"{e2d2_confirm_ret_panel.shape[1]} tickers "
          f"({e2d2_confirm_ret_panel.index.min()} .. {e2d2_confirm_ret_panel.index.max()})")

    e2d2_confirm = run_e2_confirmation(
        e2d2_confirm_ret_panel, e2d2_shortlist, lags=E2D2_LAGS, threshold_std=E2D2_THRESHOLD_STD,
        min_events=E2D2_MIN_EVENTS,
    )
    n_confirmed = int(e2d2_confirm["confirmed"].sum())
    print(f"\n{len(e2d2_confirm)} discovery hypotheses re-tested, {n_confirmed} confirmed "
          f"(BH q<0.05 within this confirmation family)")

e2d2_confirm


## 3. Pick a config and run

Edit `CONFIG_PATH` to point at the config you want to run.

In [ ]:
CONFIG_PATH = "configs/basket_gate_10m_2023_univariate.yaml"  # <-- edit me: see full list below
# Sector-basket experiment configs: sector_basket_{oil_gas,metals_mining,financials,
# utilities}_{multivariate,univariate}.yaml -- 8 configs, 4 sectors x 2 arms. Run each pair
# (multivariate + univariate for a sector) back to back, then run mcnemar_gate_test (§13) on
# the two preds.parquet outputs to get that sector's PASS/FAIL. See
# scratchpads/phase_sector_scratch_pad.md (kept locally, not in this repo) for the full
# pre-registration (sector classification, coverage check, why tech/retail/transport were
# dropped, success criterion).
#
# Prior results, for context (all runs on raw or close_adj, daily/1h unless noted; full
# numbers in FINDINGS.md):
# - Basket gate (multivariate vs. univariate, cross-sector basket): GATE FAILED -- both arms
#   chance-level, 0/64 BH-significant cells, aggregate ΔDA ≈ 0. A follow-up per-window
#   clustering check also found no hidden localized multivariate edge (p=0.22,
#   label-permutation test) -- this is exactly what motivated the sector-basket experiment
#   above (does a more economically homogeneous basket behave differently from the
#   deliberately cross-sector basket gate).
# - Lead-lag confirmation, daily: GATE FAILED (20 discovery-shortlisted pairs, 0/20 replicated
#   out-of-sample). Caught and fixed a market-factor-residualization bug in the discovery step
#   along the way (an unresidualized run had produced an artifactual same-lag-heavy
#   shortlist).
# - Lead-lag confirmation, 1h follow-on (multivariate): GATE FAILED (13 discovery-shortlisted
#   pairs, 0/13 replicated out-of-sample).
# - Lead-lag confirmation, 1h follow-on (univariate-with-covariates): companion arm, same
#   tickers/dates but group_mode=univariate -- tests whether Chronos extracts any signal
#   through the covariate panel alone (no cross-ticker attention). Also GATE FAILED.
# - Event-conditioned burst detection (§2.2-§2.7 above, daily+1h, 3 parameter sets): all NULL.

summary = run_stage(CONFIG_PATH)

### 3.1 Batch-run: basket gate + lead-lag confirmation, context_len=35 (unattended)

Runs several configs back-to-back in one call, reusing a single loaded Chronos-2
pipeline across all of them (loading it once, not once per config -- the model load
itself is a meaningful fraction of a short config's total time). Skips any `stage_id`
whose `summary.json` already exists (so re-running this cell after an interruption
or to add new configs to the list doesn't redo finished work) unless
`FORCE_RERUN=True`. Catches and logs any single config's exception so one bad/failing
config doesn't abort the rest of the batch -- prints a final PASS/FAIL table.

Edit `BATCH_CONFIG_PATHS` below to the list you want to run, e.g. the 22
`context_len=35` follow-on configs (2026-09-17 session).

In [ ]:
BATCH_CONFIG_PATHS = [
    # Basket gate, context_len=35 (daily + 1h, both arms)
    "configs/basket_gate_daily_adj_ctx35_multivariate.yaml",
    "configs/basket_gate_daily_adj_ctx35_univariate.yaml",
    "configs/basket_gate_1h_adj_ctx35_multivariate.yaml",
    "configs/basket_gate_1h_adj_ctx35_univariate.yaml",
    # Lead-lag confirmation, context_len=35 (daily + 1h)
    "configs/leadlag_confirm_adj_ctx35.yaml",
    "configs/leadlag_confirm_1h_adj_ctx35.yaml",
    # Sector baskets, context_len=35, daily (4 sectors x 2 arms)
    "configs/sector_basket_oil_gas_ctx35_multivariate.yaml",
    "configs/sector_basket_oil_gas_ctx35_univariate.yaml",
    "configs/sector_basket_metals_mining_ctx35_multivariate.yaml",
    "configs/sector_basket_metals_mining_ctx35_univariate.yaml",
    "configs/sector_basket_financials_ctx35_multivariate.yaml",
    "configs/sector_basket_financials_ctx35_univariate.yaml",
    "configs/sector_basket_utilities_ctx35_multivariate.yaml",
    "configs/sector_basket_utilities_ctx35_univariate.yaml",
    # Sector baskets, context_len=35, 1h (4 sectors x 2 arms)
    "configs/sector_basket_oil_gas_1h_ctx35_multivariate.yaml",
    "configs/sector_basket_oil_gas_1h_ctx35_univariate.yaml",
    "configs/sector_basket_metals_mining_1h_ctx35_multivariate.yaml",
    "configs/sector_basket_metals_mining_1h_ctx35_univariate.yaml",
    "configs/sector_basket_financials_1h_ctx35_multivariate.yaml",
    "configs/sector_basket_financials_1h_ctx35_univariate.yaml",
    "configs/sector_basket_utilities_1h_ctx35_multivariate.yaml",
    "configs/sector_basket_utilities_1h_ctx35_univariate.yaml",
]
FORCE_RERUN = False  # True re-runs a config even if its summary.json already exists

import time as _time

_pipeline = None
_results = []  # (stage_id, status, detail)

for _cfg_path in BATCH_CONFIG_PATHS:
    _cfg_peek = load_config(_cfg_path)
    _stage_id = _cfg_peek["stage_id"]
    _summary_path = Path(_cfg_peek["output_dir"]) / _stage_id / "summary.json"
    if _summary_path.exists() and not FORCE_RERUN:
        print(f"[skip] {_stage_id}: summary.json already exists")
        _results.append((_stage_id, "skipped", str(_summary_path)))
        continue
    print(f"[run]  {_stage_id} ({_cfg_path}) ...")
    _t0 = _time.time()
    try:
        if _pipeline is None:
            from chronos import Chronos2Pipeline
            _pipeline = Chronos2Pipeline.from_pretrained(
                "amazon/chronos-2", device_map=DEVICE, torch_dtype=DTYPE)
        _summary = run_stage(_cfg_path, pipeline=_pipeline)
        _dt = _time.time() - _t0
        print(f"[done] {_stage_id} in {_dt:.0f}s")
        _results.append((_stage_id, "done", f"{_dt:.0f}s"))
    except Exception as _e:
        _dt = _time.time() - _t0
        print(f"[FAIL] {_stage_id} after {_dt:.0f}s: {type(_e).__name__}: {_e}")
        _results.append((_stage_id, "FAILED", f"{type(_e).__name__}: {_e}"))
        continue  # one bad config must not abort the rest of the batch

print()
print("=== batch summary ===")
for _sid, _status, _detail in _results:
    print(f"  {_status:8s} {_sid:50s} {_detail}")
_n_failed = sum(1 for _, s, _ in _results if s == "FAILED")
if _n_failed:
    print(f"\n{_n_failed} config(s) FAILED -- see messages above, re-run this cell after"
          f" fixing (completed ones will be skipped automatically).")


### 3.2 Batch-run: sector basket experiments, context_len=250 (daily + 1h, close_adj)

Same unattended batch runner as §3.1 (reuses a single loaded pipeline, skips
already-completed `stage_id`s via `summary.json`, logs failures without aborting
the batch) applied to the 16 sector-basket configs at the ORIGINAL context_len=250
(not the ctx35 follow-on in §3.1) -- 4 sectors x {daily, 1h} x {multivariate,
univariate}. The 8 daily configs are this project's first close_adj run for these
sectors (converted in place 2026-09-18, never run on raw close); the 8 1h configs are
brand new (2026-09-18), reusing the basket-gate/lead-lag confirmation screens' validated 1h window. See
`scratchpads/phase_sector_scratch_pad.md` (kept locally, not in this repo) for the full pre-registration.

In [ ]:
SECTOR_BATCH_CONFIG_PATHS = [
    # Sector baskets, context_len=250, daily, close_adj (4 sectors x 2 arms)
    "configs/sector_basket_oil_gas_multivariate.yaml",
    "configs/sector_basket_oil_gas_univariate.yaml",
    "configs/sector_basket_metals_mining_multivariate.yaml",
    "configs/sector_basket_metals_mining_univariate.yaml",
    "configs/sector_basket_financials_multivariate.yaml",
    "configs/sector_basket_financials_univariate.yaml",
    "configs/sector_basket_utilities_multivariate.yaml",
    "configs/sector_basket_utilities_univariate.yaml",
    # Sector baskets, context_len=250, 1h, close_adj (4 sectors x 2 arms)
    "configs/sector_basket_oil_gas_1h_multivariate.yaml",
    "configs/sector_basket_oil_gas_1h_univariate.yaml",
    "configs/sector_basket_metals_mining_1h_multivariate.yaml",
    "configs/sector_basket_metals_mining_1h_univariate.yaml",
    "configs/sector_basket_financials_1h_multivariate.yaml",
    "configs/sector_basket_financials_1h_univariate.yaml",
    "configs/sector_basket_utilities_1h_multivariate.yaml",
    "configs/sector_basket_utilities_1h_univariate.yaml",
]
SECTOR_FORCE_RERUN = False  # True re-runs a config even if its summary.json already exists

import time as _time

_sector_pipeline = None
_sector_results = []  # (stage_id, status, detail)

for _cfg_path in SECTOR_BATCH_CONFIG_PATHS:
    _cfg_peek = load_config(_cfg_path)
    _stage_id = _cfg_peek["stage_id"]
    _summary_path = Path(_cfg_peek["output_dir"]) / _stage_id / "summary.json"
    if _summary_path.exists() and not SECTOR_FORCE_RERUN:
        print(f"[skip] {_stage_id}: summary.json already exists")
        _sector_results.append((_stage_id, "skipped", str(_summary_path)))
        continue
    print(f"[run]  {_stage_id} ({_cfg_path}) ...")
    _t0 = _time.time()
    try:
        if _sector_pipeline is None:
            from chronos import Chronos2Pipeline
            _sector_pipeline = Chronos2Pipeline.from_pretrained(
                "amazon/chronos-2", device_map=DEVICE, torch_dtype=DTYPE)
        _summary = run_stage(_cfg_path, pipeline=_sector_pipeline)
        _dt = _time.time() - _t0
        print(f"[done] {_stage_id} in {_dt:.0f}s")
        _sector_results.append((_stage_id, "done", f"{_dt:.0f}s"))
    except Exception as _e:
        _dt = _time.time() - _t0
        print(f"[FAIL] {_stage_id} after {_dt:.0f}s: {type(_e).__name__}: {_e}")
        _sector_results.append((_stage_id, "FAILED", f"{type(_e).__name__}: {_e}"))
        continue  # one bad config must not abort the rest of the batch

print()
print("=== sector batch summary ===")
for _sid, _status, _detail in _sector_results:
    print(f"  {_status:8s} {_sid:50s} {_detail}")
_n_failed = sum(1 for _, s, _ in _sector_results if s == "FAILED")
if _n_failed:
    print(f"\n{_n_failed} config(s) FAILED -- see messages above, re-run this cell after"
          f" fixing (completed ones will be skipped automatically).")


### 3.3 Batch-run: context_len=100 gap-filling sweep (basket gate, lead-lag confirmation, sectors)

**Run this AFTER the main repo has been pushed to GitHub** (per explicit sequencing
decided 2026-09-18) -- it's an independent addition to the existing context-length
sweep, not a blocker for publishing. Same unattended batch runner as §3.1/§3.2, applied to
the 22 `context_len=100` configs -- the middle data point for a LOG-SPACED 3-point sweep:
35 -> 100 -> 250 (geometric midpoint of 35 and 250 is sqrt(35*250)~=93; 100 is the
nearest round number). Replaces an earlier context_len=60 choice, which sat closer to 35
than to 250 on a log scale and would have made an uneven-looking sweep in the writeup.
Reuses the exact same already-cached date ranges as §3.1/§3.2, only `context_len` differs.

In [ ]:
BATCH100_CONFIG_PATHS = [
    # Basket gate, context_len=100 (daily + 1h, both arms)
    "configs/basket_gate_daily_adj_ctx100_multivariate.yaml",
    "configs/basket_gate_daily_adj_ctx100_univariate.yaml",
    "configs/basket_gate_1h_adj_ctx100_multivariate.yaml",
    "configs/basket_gate_1h_adj_ctx100_univariate.yaml",
    # Lead-lag confirmation, context_len=100 (daily + 1h)
    "configs/leadlag_confirm_adj_ctx100.yaml",
    "configs/leadlag_confirm_1h_adj_ctx100.yaml",
    # Sector baskets, context_len=100, daily (4 sectors x 2 arms)
    "configs/sector_basket_oil_gas_ctx100_multivariate.yaml",
    "configs/sector_basket_oil_gas_ctx100_univariate.yaml",
    "configs/sector_basket_metals_mining_ctx100_multivariate.yaml",
    "configs/sector_basket_metals_mining_ctx100_univariate.yaml",
    "configs/sector_basket_financials_ctx100_multivariate.yaml",
    "configs/sector_basket_financials_ctx100_univariate.yaml",
    "configs/sector_basket_utilities_ctx100_multivariate.yaml",
    "configs/sector_basket_utilities_ctx100_univariate.yaml",
    # Sector baskets, context_len=100, 1h (4 sectors x 2 arms)
    "configs/sector_basket_oil_gas_1h_ctx100_multivariate.yaml",
    "configs/sector_basket_oil_gas_1h_ctx100_univariate.yaml",
    "configs/sector_basket_metals_mining_1h_ctx100_multivariate.yaml",
    "configs/sector_basket_metals_mining_1h_ctx100_univariate.yaml",
    "configs/sector_basket_financials_1h_ctx100_multivariate.yaml",
    "configs/sector_basket_financials_1h_ctx100_univariate.yaml",
    "configs/sector_basket_utilities_1h_ctx100_multivariate.yaml",
    "configs/sector_basket_utilities_1h_ctx100_univariate.yaml",
]
BATCH100_FORCE_RERUN = False  # True re-runs a config even if its summary.json already exists

import time as _time

_batch100_pipeline = None
_batch100_results = []  # (stage_id, status, detail)

for _cfg_path in BATCH100_CONFIG_PATHS:
    _cfg_peek = load_config(_cfg_path)
    _stage_id = _cfg_peek["stage_id"]
    _summary_path = Path(_cfg_peek["output_dir"]) / _stage_id / "summary.json"
    if _summary_path.exists() and not BATCH100_FORCE_RERUN:
        print(f"[skip] {_stage_id}: summary.json already exists")
        _batch100_results.append((_stage_id, "skipped", str(_summary_path)))
        continue
    print(f"[run]  {_stage_id} ({_cfg_path}) ...")
    _t0 = _time.time()
    try:
        if _batch100_pipeline is None:
            from chronos import Chronos2Pipeline
            _batch100_pipeline = Chronos2Pipeline.from_pretrained(
                "amazon/chronos-2", device_map=DEVICE, torch_dtype=DTYPE)
        _summary = run_stage(_cfg_path, pipeline=_batch100_pipeline)
        _dt = _time.time() - _t0
        print(f"[done] {_stage_id} in {_dt:.0f}s")
        _batch100_results.append((_stage_id, "done", f"{_dt:.0f}s"))
    except Exception as _e:
        _dt = _time.time() - _t0
        print(f"[FAIL] {_stage_id} after {_dt:.0f}s: {type(_e).__name__}: {_e}")
        _batch100_results.append((_stage_id, "FAILED", f"{type(_e).__name__}: {_e}"))
        continue  # one bad config must not abort the rest of the batch

print()
print("=== context_len=100 batch summary ===")
for _sid, _status, _detail in _batch100_results:
    print(f"  {_status:8s} {_sid:50s} {_detail}")
_n_failed = sum(1 for _, s, _ in _batch100_results if s == "FAILED")
if _n_failed:
    print(f"\n{_n_failed} config(s) FAILED -- see messages above, re-run this cell after"
          f" fixing (completed ones will be skipped automatically).")


## 4. Inspect outputs in-line

After `run_stage` finishes, the cell below loads the metrics tables for quick review without leaving the notebook.

In [ ]:
cfg = load_config(CONFIG_PATH)
stage_out = Path(cfg["output_dir"]) / cfg["stage_id"]
print("outputs at:", stage_out)
print("files:", sorted(p.name for p in stage_out.rglob("*") if p.is_file()))

import pandas as pd
metrics = pd.read_csv(stage_out / "metrics.csv")
agg     = pd.read_csv(stage_out / "metrics_aggregate.csv")
print("\n--- aggregate ---");        print(agg.to_string(index=False))
print("\n--- per-cell (head) ---");  print(metrics.head(20).to_string(index=False))

# Display every plot saved by run_stage
from IPython.display import Image, display
for p in sorted((stage_out / "plots").glob("*.png")):
    print(p.name); display(Image(str(p)))


## 5. Notes

- **Cache only**: `run_stage` reads from Parquet cache. If a file is missing it errors loudly with the exact missing key — fix the config / re-run §2 prefetch.
- **Multiple runs in one session**: just re-edit `CONFIG_PATH` and re-run §3 + §4. Outputs are namespaced by `stage_id`; nothing is overwritten across runs.
- **Long runs**: `run_walk_forward` checkpoints `preds_partial.parquet` every 25 windows.
- **finetuning_wip**: when fine-tuning is wired in, it lives behind a `cfg["path_b"]["enabled"]` flag — same runner.